# Session 08 — IT Helpdesk Chatbot with RAG

**NLP Engineering Course · Final Session**

---

## Objectives

By the end of this session you will have built a **working IT helpdesk chatbot** that:

1. Searches a knowledge base of resolved IT tickets using two retrieval strategies (TF-IDF and sentence embeddings)
2. Synthesises a grounded answer using the **Groq API** (llama-3.3-70b)
3. Exposes a chat interface via **Gradio**, deployable with a single call

## Structure

| Part | Content | Approx. time |
|------|---------|-------------|
| **Part 1** | Setup · Dataset · EDA · TF-IDF retriever | 09:40 → 11:15 |
| **Part 2** | Embedding retriever · Comparison · Groq API · ChatBot class | 11:15 → 12:15 |
| **Part 3** | Gradio UI · Deploy | 12:15 → 12:45 |

## Code conventions

- Every function and class must have a **numpy-style docstring**.
- One function = one responsibility. Keep functions short and testable.
- Stub functions are marked with `# YOUR CODE HERE`. Do not change signatures.

---

## Part 1 · Setup, Dataset, EDA, TF-IDF Retriever

### 1.1 Install dependencies

Run the cell below **once**. If you are on Google Colab, the runtime will restart automatically after pip installs -- just re-run all cells from the top.

In [ ]:
# Install all required packages
# kagglehub  : programmatic Kaggle dataset download (no CLI required)
# openai    : OpenAI-compatible SDK, used with Groq's endpoint
# sentence-transformers : pre-trained sentence embedding models
# gradio     : instant web UI

%pip install -q kagglehub openai sentence-transformers gradio

### 1.2 Configure the Groq API key

You need a Groq API key. Get one free (no credit card) at https://console.groq.com

**Option A — Google Colab (recommended)**
1. Open the 🔑 **Secrets** panel on the left sidebar (key icon).
2. Add a secret named `GROQ_API_KEY` with your key as the value.
3. Toggle **Notebook access** to ON.
4. Run the cell below -- it reads the key automatically.

**Option B — Local Jupyter**
Paste your key directly into the `GROQ_API_KEY` variable below.
**Never commit a real key to git.**

> **📌 Note:** The API key is used only in Part 2. You can complete Part 1 without it.

In [ ]:
import os

# --- Option A: Colab secrets (preferred) ---
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception:
    # --- Option B: paste your key here (local Jupyter only) ---
    GROQ_API_KEY = "gsk_..."   # <-- replace with your key
    print('API key set manually (make sure not to commit this).')

# Sanity check
if not GROQ_API_KEY or GROQ_API_KEY.startswith('gsk_...'):
    print('WARNING: API key not configured. Groq calls will fail in Part 2.')
else:
    print(f'Key starts with: {GROQ_API_KEY[:12]}...')

### 1.3 Download the dataset

We use the **Tech Support Conversations Dataset** from Kaggle:
https://www.kaggle.com/datasets/steve1215rogg/tech-support-conversations-dataset

**First-time setup (do this once):**

1. Create a free account at https://www.kaggle.com
2. Go to *Your profile > Settings > API > Create New Token*
3. A file `kaggle.json` is downloaded. Upload it to Colab or place it at `~/.kaggle/kaggle.json`.

The cell below uses `kagglehub` which reads `kaggle.json` automatically.

In [ ]:
import kagglehub
import pandas as pd
import os

# Download dataset (cached after first run)
path = kagglehub.dataset_download(
    'steve1215rogg/tech-support-conversations-dataset'
)
print(f'Dataset downloaded to: {path}')

# Find the CSV file inside the downloaded folder
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'CSV files found: {csv_files}')

df_raw = pd.read_csv(os.path.join(path, csv_files[0]))
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

### 1.4 Exploratory Data Analysis

Before building anything, understand what you have.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print('=== Schema ===')
print(df_raw.dtypes)
print()
print('=== Null values ===')
print(df_raw.isnull().sum())
print()
print('=== Issue_Status distribution ===')
print(df_raw['Issue_Status'].value_counts())
print()
print('=== Issue_Category distribution ===')
print(df_raw['Issue_Category'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Status pie chart
df_raw['Issue_Status'].value_counts().plot(
    kind='pie', ax=axes[0], autopct='%1.1f%%',
    colors=['#3fb950', '#e3b341', '#f78166']
)
axes[0].set_title('Issue Status')
axes[0].set_ylabel('')

# Category bar chart
df_raw['Issue_Category'].value_counts().plot(
    kind='barh', ax=axes[1], color='#58a6ff'
)
axes[1].set_title('Tickets per Category')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

**Question 1.4** — What percentage of tickets are resolved? Why is filtering important for our use case?

*Your answer here*

### 1.5 Load and filter the knowledge base

In [ ]:
def load_and_filter_tickets(df: pd.DataFrame) -> pd.DataFrame:
    """Load and filter tickets to keep only resolved ones.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe with columns: Conversation_ID, Customer_Issue,
        Tech_Response, Resolution_Time, Issue_Category, Issue_Status.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe with an additional column `kb_text` that
        concatenates Issue_Category and Customer_Issue for indexing.
        Index is reset.

    Notes
    -----
    Only tickets with Issue_Status == 'Resolved' are kept.
    The `kb_text` column is the text that will be indexed by the retriever:
    prepending the category improves retrieval for category-specific queries.

    Examples
    --------
    >>> filtered = load_and_filter_tickets(df_raw)
    >>> 'Resolved' in filtered['Issue_Status'].unique()
    True
    >>> 'kb_text' in filtered.columns
    True
    """
    # YOUR CODE HERE
    pass


tickets = load_and_filter_tickets(df_raw)
print(f'Knowledge base: {len(tickets)} resolved tickets')
print(tickets[['Issue_Category', 'kb_text']].head(3))

### 1.6 TF-IDF Retriever

We build a retriever as a **class** with two methods:
- `fit(texts)` — build the TF-IDF matrix from the knowledge base
- `retrieve(query, k)` — return the top-k most similar tickets

This interface will be shared with the embedding retriever in Part 2, making them interchangeable.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


class TfidfRetriever:
    """Sparse retriever based on TF-IDF and cosine similarity.

    Parameters
    ----------
    max_features : int, optional
        Maximum vocabulary size for the TF-IDF vectorizer. Default 10_000.
    ngram_range : tuple of int, optional
        The (min, max) n-gram range. Default (1, 2).

    Attributes
    ----------
    vectorizer_ : TfidfVectorizer
        Fitted vectorizer.
    index_ : scipy.sparse matrix of shape (n_docs, n_features)
        TF-IDF matrix of the knowledge base.
    """

    def __init__(self, max_features: int = 10_000, ngram_range: tuple = (1, 2)):
        # YOUR CODE HERE
        pass

    def fit(self, texts: list) -> 'TfidfRetriever':
        """Fit the vectorizer and build the TF-IDF index.

        Parameters
        ----------
        texts : list of str
            The knowledge base texts to index.

        Returns
        -------
        TfidfRetriever
            Self (for method chaining).
        """
        # YOUR CODE HERE
        pass

    def retrieve(self, query: str, k: int = 3) -> list:
        """Retrieve the top-k most similar documents.

        Parameters
        ----------
        query : str
            The user's query.
        k : int, optional
            Number of results to return. Default 3.

        Returns
        -------
        list of dict
            Each dict has keys: `index` (int), `score` (float),
            where `index` is the row position in the knowledge base dataframe.

        Examples
        --------
        >>> retriever = TfidfRetriever().fit(tickets['kb_text'].tolist())
        >>> results = retriever.retrieve('my laptop won\'t connect to wifi', k=3)
        >>> len(results)
        3
        >>> all('score' in r for r in results)
        True
        """
        # YOUR CODE HERE
        pass

In [ ]:
# Build and test the TF-IDF retriever
tfidf_retriever = TfidfRetriever(max_features=10_000, ngram_range=(1, 2))
tfidf_retriever.fit(tickets['kb_text'].tolist())

print(f'Index shape: {tfidf_retriever.index_.shape}')
print(f'Vocabulary size: {len(tfidf_retriever.vectorizer_.vocabulary_)}')

In [ ]:
def show_retrieval_results(query: str, retriever, tickets: pd.DataFrame, k: int = 3) -> None:
    """Pretty-print retrieval results for a given query.

    Parameters
    ----------
    query : str
        The user query.
    retriever : TfidfRetriever or EmbeddingRetriever
        A fitted retriever.
    tickets : pd.DataFrame
        The knowledge base dataframe.
    k : int, optional
        Number of results to show. Default 3.
    """
    results = retriever.retrieve(query, k=k)
    print(f'Query: "{query}"')
    print('-' * 60)
    for i, r in enumerate(results):
        row = tickets.iloc[r['index']]
        print(f'[{i+1}] score={r["score"]:.3f}  category={row["Issue_Category"]}')
        print(f'     Issue   : {row["Customer_Issue"][:90]}...')
        print(f'     Solution: {row["Tech_Response"][:90]}...')
        print()

In [ ]:
# Test with 5 queries -- observe the results
test_queries = [
    "my laptop won't connect to wifi",
    "printer not working after driver update",
    "I forgot my password and can't log in",
    "computer is extremely slow and freezing",
    "software keeps crashing on startup",
]

for q in test_queries:
    show_retrieval_results(q, tfidf_retriever, tickets, k=3)
    print('=' * 60)

**Question 1.6** — For which queries does TF-IDF struggle? Can you find a query that returns irrelevant results due to vocabulary mismatch?

*Your answer here*

---
## Part 2 · Embedding Retriever · Comparison · Groq API · ChatBot Class

### 2.1 Embedding Retriever

We use `all-MiniLM-L6-v2` from `sentence-transformers`:
- 384-dimensional dense vectors
- 80 MB, runs on CPU in seconds
- Trained specifically for semantic similarity tasks

The class must expose the **same interface** as `TfidfRetriever` (`fit` / `retrieve`).

In [ ]:
from sentence_transformers import SentenceTransformer


class EmbeddingRetriever:
    """Dense retriever based on sentence embeddings and cosine similarity.

    Parameters
    ----------
    model_name : str, optional
        HuggingFace model identifier for SentenceTransformer.
        Default 'all-MiniLM-L6-v2'.

    Attributes
    ----------
    model_ : SentenceTransformer
        Loaded embedding model.
    index_ : np.ndarray of shape (n_docs, embedding_dim)
        L2-normalised embeddings of the knowledge base.
    """

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        # YOUR CODE HERE
        pass

    def fit(self, texts: list, batch_size: int = 64, show_progress: bool = True) -> 'EmbeddingRetriever':
        """Encode all texts and build the embedding index.

        Parameters
        ----------
        texts : list of str
            Knowledge base texts to encode.
        batch_size : int, optional
            Encoding batch size. Default 64.
        show_progress : bool, optional
            Show tqdm progress bar. Default True.

        Returns
        -------
        EmbeddingRetriever
            Self (for method chaining).

        Notes
        -----
        Embeddings are L2-normalised so that dot product equals cosine similarity.
        """
        # YOUR CODE HERE
        pass

    def retrieve(self, query: str, k: int = 3) -> list:
        """Retrieve the top-k most similar documents.

        Parameters
        ----------
        query : str
            The user's query.
        k : int, optional
            Number of results to return. Default 3.

        Returns
        -------
        list of dict
            Each dict has keys: `index` (int), `score` (float).
        """
        # YOUR CODE HERE
        pass

In [ ]:
# Build the embedding index (may take ~30 s on CPU)
emb_retriever = EmbeddingRetriever(model_name='all-MiniLM-L6-v2')
emb_retriever.fit(tickets['kb_text'].tolist())

print(f'Index shape: {emb_retriever.index_.shape}')

### 2.2 Head-to-head comparison

Use the same 5 queries from Part 1. For each, run both retrievers and compare the results.

In [ ]:
comparison_queries = [
    "network connectivity issue after OS upgrade",       # semantic paraphrase of WiFi query
    "HP device became unresponsive after patch install",  # zero lexical overlap with printer tickets
    "account authentication is broken",
    "machine is sluggish and hangs",
    "application fails to open",
]

for q in comparison_queries:
    print(f'QUERY: "{q}"')
    print('--- TF-IDF ---')
    show_retrieval_results(q, tfidf_retriever, tickets, k=2)
    print('--- Embeddings ---')
    show_retrieval_results(q, emb_retriever, tickets, k=2)
    print('=' * 70)

**Question 2.2** — Pick two queries where the two retrievers give different top-1 results. Which retriever is correct, and why?

*Your answer here*

### 2.3 Prompt builder

The prompt builder formats the retrieved tickets and the user query into a structured prompt for the LLM.

In [ ]:
SYSTEM_PROMPT = """You are an IT support assistant at a company helpdesk.
Your role is to help employees resolve technical issues by themselves before opening a new ticket.

Rules:
1. Answer ONLY using the resolved tickets provided as context.
2. If the context does not contain enough information, say so clearly and suggest opening a ticket.
3. Be concise, practical, and step-by-step when describing solutions.
4. Always mention the source ticket category at the end of your answer.
"""


def build_prompt(query: str, retrieved_tickets: list, tickets_df: pd.DataFrame) -> str:
    """Build the user-turn prompt from a query and retrieved ticket indices.

    Parameters
    ----------
    query : str
        The user's question.
    retrieved_tickets : list of dict
        Output of retriever.retrieve(), each dict has 'index' and 'score'.
    tickets_df : pd.DataFrame
        The knowledge base dataframe.

    Returns
    -------
    str
        Formatted prompt string containing context and user question.

    Examples
    --------
    >>> retrieved = [{'index': 0, 'score': 0.9}]
    >>> prompt = build_prompt('WiFi broken', retrieved, tickets)
    >>> 'CONTEXT' in prompt
    True
    >>> 'WiFi broken' in prompt
    True
    """
    context_parts = []
    for i, r in enumerate(retrieved_tickets):
        row = tickets_df.iloc[r['index']]
        context_parts.append(
            f"--- Resolved Ticket {i+1} (Category: {row['Issue_Category']}) ---\n"
            f"Issue: {row['Customer_Issue']}\n"
            f"Solution: {row['Tech_Response']}"
        )
    context = "\n\n".join(context_parts)

    # YOUR CODE HERE
    # Build the final prompt string combining context and query
    pass


# Quick test
sample_results = tfidf_retriever.retrieve("wifi not working", k=2)
sample_prompt = build_prompt("wifi not working", sample_results, tickets)
print(sample_prompt)

### 2.4 Groq API call

In [ ]:
from openai import OpenAI

GROQ_MODEL = "llama-3.3-70b-versatile"


def ask_groq(
    user_prompt: str,
    api_key: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = GROQ_MODEL,
    max_tokens: int = 512,
) -> str:
    """Send a prompt to the Groq API and return the text response.

    Parameters
    ----------
    user_prompt : str
        The user-turn message (context + question, built by build_prompt).
    api_key : str
        Groq API key.
    system_prompt : str, optional
        System instruction (Groq system role). Default SYSTEM_PROMPT.
    model : str, optional
        Groq model identifier. Default GROQ_MODEL.
    max_tokens : int, optional
        Maximum tokens in the response. Default 512.

    Returns
    -------
    str
        The text of the model's response.

    Raises
    ------
    openai.APIError
        If the API call fails (invalid key, quota exceeded, etc.).

    Examples
    --------
    >>> response = ask_groq('How do I fix WiFi?', api_key=GROQ_API_KEY)
    >>> isinstance(response, str)
    True
    """
    # YOUR CODE HERE
    pass


# Quick end-to-end test
test_query = "My laptop won't connect to WiFi after a Windows update."
results = emb_retriever.retrieve(test_query, k=3)
prompt = build_prompt(test_query, results, tickets)
answer = ask_groq(prompt, api_key=GROQ_API_KEY)
print(answer)

### 2.5 HelpDeskChatBot class

Assemble everything into a single class with a clean `answer()` interface.

In [ ]:
class HelpDeskChatBot:
    """IT helpdesk chatbot using retrieval-augmented generation.

    Combines a retriever (TF-IDF or embeddings) with the Groq API
    to answer IT support questions grounded in resolved tickets.

    Parameters
    ----------
    tickets : pd.DataFrame
        Resolved tickets dataframe with columns: Customer_Issue,
        Tech_Response, Issue_Category, kb_text.
    retriever : TfidfRetriever or EmbeddingRetriever
        A fitted retriever exposing .retrieve(query, k) -> list of dict.
    api_key : str
        Groq API key.
    k : int, optional
        Number of tickets to retrieve per query. Default 3.

    Examples
    --------
    >>> bot = HelpDeskChatBot(tickets, emb_retriever, GROQ_API_KEY)
    >>> result = bot.answer('WiFi not working')
    >>> 'response' in result and 'sources' in result
    True
    """

    def __init__(self, tickets: pd.DataFrame, retriever, api_key: str, k: int = 3):
        # YOUR CODE HERE
        pass

    def answer(self, query: str) -> dict:
        """Answer a user query using retrieved tickets via the Groq API.

        Parameters
        ----------
        query : str
            The user's IT support question.

        Returns
        -------
        dict
            Dictionary with keys:
            - 'response' (str): The model's answer.
            - 'sources' (list of dict): Retrieved tickets, each with
              keys 'category', 'issue', 'solution', 'score'.
        """
        # YOUR CODE HERE
        pass


# Instantiate and test
bot = HelpDeskChatBot(tickets, emb_retriever, GROQ_API_KEY, k=3)

result = bot.answer("My screen goes black randomly and the laptop restarts.")
print('=== RESPONSE ===')
print(result['response'])
print()
print('=== SOURCES ===')
for s in result['sources']:
    print(f"[{s['category']}] score={s['score']:.3f}  {s['issue'][:70]}...")

---
## Part 3 · Gradio Chat Interface

### 3.1 Basic Gradio ChatInterface

Gradio's `ChatInterface` wraps any function `fn(message, history) -> str` into a full chat UI.
The `history` argument is a list of `(user_message, bot_response)` tuples automatically managed by Gradio.

In [ ]:
import gradio as gr


def format_sources(sources: list) -> str:
    """Format retrieved ticket sources into a readable string.

    Parameters
    ----------
    sources : list of dict
        List of source dicts with keys 'category', 'issue', 'solution', 'score'.

    Returns
    -------
    str
        Markdown-formatted string listing each source.
    """
    if not sources:
        return ''
    lines = ['\n\n---\n📎 **Resolved tickets used:**']
    for i, s in enumerate(sources):
        lines.append(f"`{i+1}.` [{s['category']}] {s['issue'][:80]}...  *(score: {s['score']:.2f})*")
    return '\n'.join(lines)


def chat_fn(user_message: str, history: list) -> str:
    """Gradio chat handler: retrieve, generate, and format the response.

    Parameters
    ----------
    user_message : str
        The latest user message.
    history : list of tuple
        Previous (user, bot) message pairs managed by Gradio.

    Returns
    -------
    str
        Full response string including answer and formatted sources.
    """
    # YOUR CODE HERE
    # 1. Call bot.answer(user_message)
    # 2. Format sources with format_sources()
    # 3. Return response + sources as a single string
    pass


demo = gr.ChatInterface(
    fn=chat_fn,
    title="🖥️ IT Helpdesk Assistant",
    description=(
        "Describe your IT issue and I'll search our knowledge base of resolved tickets.\n"
        "Try: *'My WiFi stopped working after a Windows update'*"
    ),
    examples=[
        "My laptop won't connect to WiFi",
        "Printer not responding after driver update",
        "I forgot my password and the reset email never arrived",
        "My computer is very slow and freezes randomly",
    ],
    theme=gr.themes.Soft(),
)

# Launch locally (or with share=True for a public URL in Colab)
demo.launch(share=True)

### 3.2 Bonus — History-aware follow-up questions

Currently each query is independent. Extend `chat_fn` to include the conversation history in the Groq prompt, allowing follow-up questions like:

- *User:* "My WiFi doesn't work"
- *Bot:* "Try resetting your network adapter..."
- *User:* "I tried that, it didn't help. What else?"

> **Hint:** Pass the last 2-3 turns of history as additional messages in the `client.chat.completions.create()` call.

In [ ]:
# BONUS: history-aware chat_fn
def ask_groq_with_history(
    user_prompt: str,
    history: list,
    api_key: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = GROQ_MODEL,
    max_tokens: int = 512,
    max_history_turns: int = 3,
) -> str:
    """Send a prompt to the Groq API including recent conversation history.

    Parameters
    ----------
    user_prompt : str
        Current user turn (with context injected).
    history : list of tuple
        Previous (user, bot) message pairs from Gradio.
    api_key : str
        Groq API key.
    system_prompt : str, optional
        System instruction. Default SYSTEM_PROMPT.
    model : str, optional
        Groq model. Default GROQ_MODEL.
    max_tokens : int, optional
        Maximum response tokens. Default 512.
    max_history_turns : int, optional
        Number of recent history turns to include. Default 3.

    Returns
    -------
    str
        The model's response text.
    """
    # YOUR CODE HERE
    pass

---
## Summary

You have built a complete RAG pipeline:

| Component | What it does |
|-----------|-------------|
| `load_and_filter_tickets` | Loads and cleans the knowledge base |
| `TfidfRetriever` | Sparse lexical retrieval |
| `EmbeddingRetriever` | Dense semantic retrieval |
| `build_prompt` | Structures context for the LLM |
| `ask_groq` | Calls the Groq API |
| `HelpDeskChatBot` | End-to-end orchestrator |
| `chat_fn` + Gradio | Web interface |

**What we learned over 8 sessions:**
- S01: Tokenisation is the foundation
- S02–S03: Statistical models are strong baselines (Naive Bayes, Logistic Regression)
- S04–S05: Word representations matter (TF-IDF n-grams, Word2Vec)
- S06: Neural sequence models (LSTM, attention)
- S07: Pre-trained transformers dominate (DistilBERT fine-tuning)
- S08: LLMs + retrieval = production systems
